# RQ5 — SPX 0DTE Options Data Retrieval and Contract Selection

**Research question:**  
*Can these predictions be used to construct a profitable end-of-day options trading strategy, after accounting for transaction costs and realistic execution constraints?*

This notebook is **Stage 1 of RQ5**. It does not yet calculate final strategy profitability.

It:

1. loads the out-of-fold RQ1–RQ3 predictions created for RQ4;
2. applies the pre-specified primary opportunity filter;
3. discovers same-day-expiry SPX option contracts from Massive;
4. selects ATM and optional OTM call/put contracts without using future option prices;
5. downloads one-minute option aggregate bars around the final hour;
6. saves the contract map and raw option-bar dataset for the later RQ5 backtest.

## Primary opportunity rule

The primary ML trade-candidate filter is fixed **before options P&L is examined**:

- RQ1 confidence: top 30% of outer-fold predictions;
- RQ2 predicted absolute movement: at least 20 bps;
- RQ3: large-movement signal = 1.

RQ1 determines direction:

- predicted Up → Call;
- predicted Down → Put.

The strongest simple RQ1 comparator, 60-minute mean reversion, is also stored. Since it may choose the opposite direction on the same candidate date, both calls and puts are retrieved.

## Execution-data limitation

Massive Options Starter supplies minute aggregates but not historical NBBO quote data. Therefore this notebook retrieves trade-derived OHLC/VWAP bars. The later backtest must use explicit execution-cost sensitivity assumptions rather than claim that actual historical bid/ask fills were reconstructed.

No fresh post-development holdout is loaded here.

### Retail-affordability extension

In addition to the primary ATM contract, this version retrieves contracts approximately
5, 10, 15, 20, 25 and 30 SPX points out-of-the-money. The purpose is to test whether
lower-premium contracts make the strategy more accessible to a small retail account.

These farther-OTM contracts are **sensitivity specifications**, not new primary strategies.
Their lower entry premium must be evaluated against their lower delta, greater probability
of expiring worthless, and greater dependence on a large late-session SPX move.

## 1. Imports and configuration

In [5]:
from pathlib import Path
from datetime import datetime
import json
import os
import time
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv, dotenv_values
import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

API_BASE = "https://api.massive.com"
PROJECT_ROOT = Path.cwd()
ENV_PATH = PROJECT_ROOT / "main.env"

print("Notebook working directory:", PROJECT_ROOT)
print("Expected environment file:", ENV_PATH)
print("File exists:", ENV_PATH.is_file())

if not ENV_PATH.is_file():
    print("Files containing 'env' in the project directory:")
    print([path.name for path in PROJECT_ROOT.glob("*env*")])
    raise FileNotFoundError(f"Environment file not found: {ENV_PATH}")

# Parse the file without putting its values into os.environ.
parsed = dotenv_values(ENV_PATH)

print("Parsed variable names:", list(parsed.keys()))
print("MASSIVE_API_KEY found:", "MASSIVE_API_KEY" in parsed)
print("MASSIVE_API_KEY has a value:", bool(parsed.get("MASSIVE_API_KEY")))


# Either set MASSIVE_API_KEY in your environment or paste it temporarily here.
API_KEY = os.getenv("MASSIVE_API_KEY", "").strip()
if not API_KEY:
    API_KEY = "PASTE_YOUR_MASSIVE_API_KEY_HERE"

if API_KEY == "PASTE_YOUR_MASSIVE_API_KEY_HERE":
    print("WARNING: Add your Massive API key before running API cells.")

# Frozen RQ4/RQ5 opportunity-rule settings.
PRIMARY_RQ1_COVERAGE = 0.30
PRIMARY_RQ2_MIN_BPS = 20.0

# Contract selection:
# 0 = ATM. Positive values represent approximately that many SPX points OTM.
# ATM is the primary specification. OTM variants from 5 to 30 points are
# retrieved for later affordability/capital-efficiency sensitivity analysis.
STRIKE_OFFSETS_POINTS = [0, 5, 10, 15, 20, 25, 30]

# Download a small buffer around the target trading window.
OPTION_BAR_START_ET = "14:55"
OPTION_BAR_END_ET = "16:00"

# Massive Options Starter is rolling two-year historical aggregate access.
# We do not hard-code a historical start date: it is calculated when the notebook runs.
OPTIONS_HISTORY_YEARS = 2

# API behaviour.
HTTP_TIMEOUT = 45
MAX_RETRIES = 5
RETRY_SLEEP_SECONDS = 1.5

# SPX reference contracts normally resolve under SPX; SPXW is kept as a fallback
# because same-day weekly contracts may carry SPXW-style option symbols.
REFERENCE_UNDERLYING_CANDIDATES = ["SPX", "SPXW"]

Notebook working directory: c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation
Expected environment file: c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation\main.env
File exists: True
Parsed variable names: ['MASSIVE_API_KEY']
MASSIVE_API_KEY found: True
MASSIVE_API_KEY has a value: True


## 2. Locate project files

In [ ]:
def locate_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "market.duckdb").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/market.duckdb. "
        "Run this notebook inside your dissertation database project."
    )

PROJECT_ROOT = locate_project_root(Path.cwd())

RQ4_TABLE_ROOT = (
    PROJECT_ROOT / "outputs" / "rq4_economic_meaningfulness" / "tables"
)
RQ4_COMBINED_PATH = (
    RQ4_TABLE_ROOT / "rq4_combined_outer_fold_predictions.csv"
)

RQ5_ROOT = PROJECT_ROOT / "outputs" / "rq5_options_trading"
RQ5_RAW_ROOT = RQ5_ROOT / "raw"
RQ5_TABLE_ROOT = RQ5_ROOT / "tables"

RQ5_ROOT.mkdir(parents=True, exist_ok=True)
RQ5_RAW_ROOT.mkdir(parents=True, exist_ok=True)
RQ5_TABLE_ROOT.mkdir(parents=True, exist_ok=True)

if not RQ4_COMBINED_PATH.exists():
    raise FileNotFoundError(
        f"Missing {RQ4_COMBINED_PATH}. Run rq4_economic_meaningfulness.ipynb first."
    )

print("Project root:", PROJECT_ROOT)
print("RQ4 source:", RQ4_COMBINED_PATH)
print("RQ5 output:", RQ5_ROOT)

## 3. Load RQ4 outer-fold predictions and freeze the RQ5 candidate dates

In [ ]:
combined = pd.read_csv(RQ4_COMBINED_PATH)
combined["session_date"] = pd.to_datetime(combined["session_date"])

required_cols = {
    "session_date",
    "spx_at_1500",
    "rq1_prediction",
    "rq1_confidence_percentile",
    "rq2_predicted_move_bps",
    "rq3_positive",
    "ret_last_60m",
    "actual_abs_move_bps",
    "actual_signed_move_bps",
}

missing = required_cols.difference(combined.columns)
if missing:
    raise ValueError(f"RQ4 combined file is missing columns: {sorted(missing)}")

combined["rq1_high_confidence"] = (
    combined["rq1_confidence_percentile"] > (1.0 - PRIMARY_RQ1_COVERAGE)
)

combined["primary_trade_candidate"] = (
    combined["rq1_high_confidence"]
    & (combined["rq2_predicted_move_bps"] >= PRIMARY_RQ2_MIN_BPS)
    & combined["rq3_positive"].astype(bool)
)

combined["ml_direction"] = np.where(
    combined["rq1_prediction"].astype(int).eq(1),
    "call",
    "put",
)

# 60-minute mean-reversion benchmark:
# negative prior return -> predict up/call; positive prior return -> predict down/put.
combined["mean_reversion_direction"] = np.where(
    combined["ret_last_60m"] < 0,
    "call",
    "put",
)

candidates = (
    combined[combined["primary_trade_candidate"]]
    .copy()
    .sort_values("session_date")
    .reset_index(drop=True)
)

print("Three-model intersection sessions:", len(combined))
print("Primary RQ5 candidate sessions:", len(candidates))
display(
    candidates[
        [
            "session_date",
            "spx_at_1500",
            "rq1_confidence_percentile",
            "rq2_predicted_move_bps",
            "rq3_positive",
            "ml_direction",
            "mean_reversion_direction",
            "actual_abs_move_bps",
        ]
    ]
)

candidate_path = RQ5_TABLE_ROOT / "rq5_primary_candidate_sessions.csv"
candidates.to_csv(candidate_path, index=False)
print("Saved:", candidate_path)

## 4. Respect the rolling Options Starter history window

The exact entitlement is enforced by the API. This local date filter simply prevents unnecessary requests for observations that are clearly older than the nominal two-year history window.

If a date near the boundary is unavailable, the API response is logged rather than silently dropped.

In [ ]:
today = pd.Timestamp.now(tz="America/New_York").normalize().tz_localize(None)
nominal_history_start = today - pd.DateOffset(years=OPTIONS_HISTORY_YEARS)

candidates["within_nominal_options_history"] = (
    candidates["session_date"] >= nominal_history_start
)

print("Notebook run date:", today.date())
print("Nominal two-year history start:", nominal_history_start.date())
print(
    "Candidates inside nominal options history:",
    int(candidates["within_nominal_options_history"].sum()),
    "/",
    len(candidates),
)

eligible_candidates = (
    candidates[candidates["within_nominal_options_history"]]
    .copy()
    .reset_index(drop=True)
)

display(
    candidates[
        [
            "session_date",
            "primary_trade_candidate",
            "within_nominal_options_history",
        ]
    ]
)

## 5. Massive REST helpers

In [ ]:
session = requests.Session()


def massive_get(path_or_url, params=None, allow_error=False):
    if path_or_url.startswith("http"):
        url = path_or_url
    else:
        url = API_BASE + path_or_url

    params = dict(params or {})
    params["apiKey"] = API_KEY

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                url,
                params=params,
                timeout=HTTP_TIMEOUT,
            )

            if response.status_code == 200:
                return response.json(), response.status_code

            # Retry temporary server/rate-limit failures.
            if response.status_code in {429, 500, 502, 503, 504}:
                last_error = (
                    f"HTTP {response.status_code}: {response.text[:300]}"
                )
                time.sleep(RETRY_SLEEP_SECONDS * attempt)
                continue

            if allow_error:
                try:
                    payload = response.json()
                except Exception:
                    payload = {"raw_text": response.text[:1000]}
                return payload, response.status_code

            raise RuntimeError(
                f"Massive request failed HTTP {response.status_code}: "
                f"{response.text[:1000]}"
            )

        except requests.RequestException as exc:
            last_error = repr(exc)
            time.sleep(RETRY_SLEEP_SECONDS * attempt)

    raise RuntimeError(
        f"Massive request failed after {MAX_RETRIES} attempts. "
        f"Last error: {last_error}"
    )


def fetch_all_pages(path, params):
    rows = []
    first = True
    next_url = None

    while first or next_url:
        if first:
            payload, status = massive_get(path, params=params)
            first = False
        else:
            # next_url already carries cursor/query state; API key is re-added.
            payload, status = massive_get(next_url, params={})

        if status != 200:
            raise RuntimeError(f"Unexpected HTTP status {status}")

        rows.extend(payload.get("results") or [])
        next_url = payload.get("next_url")

    return rows

## 6. Discover same-day SPX option contracts

In [ ]:
def fetch_same_day_contracts(session_date):
    date_str = pd.Timestamp(session_date).strftime("%Y-%m-%d")
    attempts = []

    # First try point-in-time reference queries.
    for underlying in REFERENCE_UNDERLYING_CANDIDATES:
        params = {
            "underlying_ticker": underlying,
            "expiration_date": date_str,
            "as_of": date_str,
            "limit": 1000,
            "sort": "strike_price",
            "order": "asc",
        }

        payload, status = massive_get(
            "/v3/reference/options/contracts",
            params=params,
            allow_error=True,
        )
        attempts.append((underlying, "as_of", status))

        if status == 200 and payload.get("results"):
            frame = pd.DataFrame(payload["results"])
            frame["reference_query_underlying"] = underlying
            frame["reference_query_mode"] = "as_of"
            return frame, attempts

    # Fallback for already-expired contracts.
    for underlying in REFERENCE_UNDERLYING_CANDIDATES:
        params = {
            "underlying_ticker": underlying,
            "expiration_date": date_str,
            "expired": "true",
            "limit": 1000,
            "sort": "strike_price",
            "order": "asc",
        }

        payload, status = massive_get(
            "/v3/reference/options/contracts",
            params=params,
            allow_error=True,
        )
        attempts.append((underlying, "expired", status))

        if status == 200 and payload.get("results"):
            frame = pd.DataFrame(payload["results"])
            frame["reference_query_underlying"] = underlying
            frame["reference_query_mode"] = "expired"
            return frame, attempts

    return pd.DataFrame(), attempts


def select_contract(
    contracts: pd.DataFrame,
    option_type: str,
    spot: float,
    otm_offset_points: float,
):
    side = contracts[
        contracts["contract_type"].astype(str).str.lower().eq(option_type)
    ].copy()

    if side.empty:
        return None

    side["strike_price"] = pd.to_numeric(
        side["strike_price"],
        errors="coerce",
    )
    side = side.dropna(subset=["strike_price"])

    if option_type == "call":
        target_strike = spot + otm_offset_points
    else:
        target_strike = spot - otm_offset_points

    side["distance_to_target"] = (
        side["strike_price"] - target_strike
    ).abs()

    # Deterministic tie-break:
    # closest target, then lower strike, then ticker.
    chosen = (
        side.sort_values(
            ["distance_to_target", "strike_price", "ticker"],
            ascending=[True, True, True],
        )
        .iloc[0]
        .to_dict()
    )

    chosen["target_strike"] = target_strike
    chosen["spot_for_selection"] = spot
    chosen["otm_offset_points"] = otm_offset_points
    return chosen

## 7. Build a dated contract-selection map

**No option price is used to choose a contract.**  
The selection uses only:

- session date;
- same-day expiry;
- option type;
- 15:00 SPX level;
- pre-specified strike offset.

This prevents contract-selection look-ahead.

In [ ]:
contract_selection_rows = []
contract_discovery_log = []

for row in eligible_candidates.itertuples(index=False):
    date = pd.Timestamp(row.session_date)
    date_str = date.strftime("%Y-%m-%d")
    spot = float(row.spx_at_1500)

    contracts, attempts = fetch_same_day_contracts(date)

    contract_discovery_log.append(
        {
            "session_date": date,
            "contracts_found": len(contracts),
            "query_attempts": repr(attempts),
        }
    )

    if contracts.empty:
        print(f"{date_str}: no same-day contracts found")
        continue

    for option_type in ["call", "put"]:
        for offset in STRIKE_OFFSETS_POINTS:
            chosen = select_contract(
                contracts=contracts,
                option_type=option_type,
                spot=spot,
                otm_offset_points=offset,
            )

            if chosen is None:
                contract_selection_rows.append(
                    {
                        "session_date": date,
                        "option_type": option_type,
                        "otm_offset_points": offset,
                        "selection_status": "no_contract",
                    }
                )
                continue

            contract_selection_rows.append(
                {
                    "session_date": date,
                    "option_type": option_type,
                    "otm_offset_points": offset,
                    "selection_status": "selected",
                    "ticker": chosen.get("ticker"),
                    "underlying_ticker": chosen.get("underlying_ticker"),
                    "expiration_date": chosen.get("expiration_date"),
                    "strike_price": chosen.get("strike_price"),
                    "target_strike": chosen.get("target_strike"),
                    "spot_for_selection": chosen.get("spot_for_selection"),
                    "strike_distance_from_spot": (
                        float(chosen.get("strike_price")) - spot
                    ),
                    "exercise_style": chosen.get("exercise_style"),
                    "shares_per_contract": chosen.get("shares_per_contract"),
                    "primary_exchange": chosen.get("primary_exchange"),
                    "reference_query_underlying": chosen.get(
                        "reference_query_underlying"
                    ),
                    "reference_query_mode": chosen.get(
                        "reference_query_mode"
                    ),
                }
            )

    print(
        f"{date_str}: {len(contracts)} contracts discovered; "
        f"selection rows added."
    )

contract_selection = pd.DataFrame(contract_selection_rows)
contract_log = pd.DataFrame(contract_discovery_log)

display(contract_selection.head(20))
display(contract_log)

contract_selection_path = RQ5_TABLE_ROOT / "rq5_contract_selection.csv"
contract_log_path = RQ5_TABLE_ROOT / "rq5_contract_discovery_log.csv"

contract_selection.to_csv(contract_selection_path, index=False)
contract_log.to_csv(contract_log_path, index=False)

print("Saved:", contract_selection_path)
print("Saved:", contract_log_path)

## 8. Download one-minute aggregate bars for selected option contracts

In [ ]:
def fetch_option_minute_bars(option_ticker, session_date):
    date_str = pd.Timestamp(session_date).strftime("%Y-%m-%d")

    path = (
        f"/v2/aggs/ticker/{option_ticker}"
        f"/range/1/minute/{date_str}/{date_str}"
    )

    params = {
        "adjusted": "true",
        "sort": "asc",
        "limit": 50000,
    }

    payload, status = massive_get(
        path,
        params=params,
        allow_error=True,
    )

    if status != 200:
        return pd.DataFrame(), status, payload

    rows = payload.get("results") or []
    if not rows:
        return pd.DataFrame(), status, payload

    frame = pd.DataFrame(rows).rename(
        columns={
            "t": "timestamp_ms",
            "o": "open",
            "h": "high",
            "l": "low",
            "c": "close",
            "v": "volume",
            "vw": "vwap",
            "n": "transactions",
        }
    )

    frame["timestamp"] = pd.to_datetime(
        frame["timestamp_ms"],
        unit="ms",
        utc=True,
    ).dt.tz_convert("America/New_York")

    frame["session_date"] = pd.Timestamp(session_date)
    frame["option_ticker"] = option_ticker

    time_text = frame["timestamp"].dt.strftime("%H:%M")
    mask = (
        (time_text >= OPTION_BAR_START_ET)
        & (time_text <= OPTION_BAR_END_ET)
    )

    return frame.loc[mask].copy(), status, payload

In [ ]:
selected_rows = contract_selection[
    contract_selection["selection_status"].eq("selected")
].copy()

# Avoid duplicate calls if a target offset resolves to the same listed strike.
unique_contracts = (
    selected_rows[
        ["session_date", "ticker"]
    ]
    .drop_duplicates()
    .sort_values(["session_date", "ticker"])
    .reset_index(drop=True)
)

bar_frames = []
bar_log_rows = []

for i, row in enumerate(unique_contracts.itertuples(index=False), start=1):
    bars, status, payload = fetch_option_minute_bars(
        row.ticker,
        row.session_date,
    )

    bar_log_rows.append(
        {
            "session_date": pd.Timestamp(row.session_date),
            "ticker": row.ticker,
            "http_status": status,
            "bars_in_window": len(bars),
            "api_status": (
                payload.get("status")
                if isinstance(payload, dict)
                else None
            ),
        }
    )

    if not bars.empty:
        bar_frames.append(bars)

    print(
        f"[{i}/{len(unique_contracts)}] "
        f"{pd.Timestamp(row.session_date).date()} {row.ticker}: "
        f"{len(bars)} bars"
    )

option_bars = (
    pd.concat(bar_frames, ignore_index=True)
    if bar_frames
    else pd.DataFrame()
)

bar_log = pd.DataFrame(bar_log_rows)

display(bar_log.head(20))
print("Downloaded option bar rows:", len(option_bars))

## 9. Attach contract metadata and candidate-signal metadata

In [ ]:
if not option_bars.empty:
    # Join each bar to every selection variant that resolved to that contract.
    option_bars_enriched = option_bars.merge(
        selected_rows,
        left_on=["session_date", "option_ticker"],
        right_on=["session_date", "ticker"],
        how="left",
        validate="many_to_many",
    )

    signal_cols = [
        "session_date",
        "spx_at_1500",
        "rq1_prediction",
        "rq1_confidence_percentile",
        "rq2_predicted_move_bps",
        "rq3_positive",
        "ml_direction",
        "mean_reversion_direction",
        "actual_abs_move_bps",
        "actual_signed_move_bps",
    ]

    option_bars_enriched = option_bars_enriched.merge(
        eligible_candidates[signal_cols],
        on="session_date",
        how="left",
        validate="many_to_one",
    )
else:
    option_bars_enriched = option_bars.copy()

display(option_bars_enriched.head(20))

## 10. Data-quality audit

A session is suitable for the later backtest only if the required contract has tradable aggregate observations around the pre-specified entry and exit windows.

The eventual backtest will not invent prices for missing minutes. It will use deterministic fallback rules that are documented before profitability is calculated.

In [ ]:
quality_rows = []

for selection in selected_rows.itertuples(index=False):
    ticker = selection.ticker
    date = pd.Timestamp(selection.session_date)

    bars = option_bars[
        (option_bars["option_ticker"].eq(ticker))
        & (option_bars["session_date"].eq(date))
    ].copy()

    if bars.empty:
        quality_rows.append(
            {
                "session_date": date,
                "ticker": ticker,
                "option_type": selection.option_type,
                "otm_offset_points": selection.otm_offset_points,
                "bars_1455_1600": 0,
                "has_1500_1505_bar": False,
                "has_1550_1600_bar": False,
                "first_bar_et": None,
                "last_bar_et": None,
                "total_volume_window": 0,
                "quality_status": "no_bars",
            }
        )
        continue

    hhmm = bars["timestamp"].dt.strftime("%H:%M")

    has_entry_window = ((hhmm >= "15:00") & (hhmm <= "15:05")).any()
    has_exit_window = ((hhmm >= "15:50") & (hhmm <= "16:00")).any()

    quality_rows.append(
        {
            "session_date": date,
            "ticker": ticker,
            "option_type": selection.option_type,
            "otm_offset_points": selection.otm_offset_points,
            "bars_1455_1600": len(bars),
            "has_1500_1505_bar": bool(has_entry_window),
            "has_1550_1600_bar": bool(has_exit_window),
            "first_bar_et": bars["timestamp"].min(),
            "last_bar_et": bars["timestamp"].max(),
            "total_volume_window": pd.to_numeric(
                bars["volume"],
                errors="coerce",
            ).sum(),
            "quality_status": (
                "usable"
                if has_entry_window and has_exit_window
                else "partial"
            ),
        }
    )

quality = pd.DataFrame(quality_rows)

display(
    quality.groupby(
        ["otm_offset_points", "option_type", "quality_status"]
    ).size().rename("contracts").to_frame()
)

display(quality.head(30))

## 11. Save RQ5 retrieval outputs

In [ ]:
bars_path = RQ5_RAW_ROOT / "rq5_option_minute_bars.parquet"
bars_csv_path = RQ5_RAW_ROOT / "rq5_option_minute_bars.csv"
bar_log_path = RQ5_TABLE_ROOT / "rq5_option_bar_download_log.csv"
quality_path = RQ5_TABLE_ROOT / "rq5_option_data_quality.csv"

if not option_bars_enriched.empty:
    # Parquet is the primary storage format.
    option_bars_enriched.to_parquet(bars_path, index=False)

    # CSV is useful for manual inspection but may be larger.
    option_bars_enriched.to_csv(bars_csv_path, index=False)

bar_log.to_csv(bar_log_path, index=False)
quality.to_csv(quality_path, index=False)

manifest = {
    "created_at_local": datetime.now().astimezone().isoformat(),
    "research_question": "RQ5",
    "stage": "options_data_retrieval",
    "fresh_holdout_used": False,
    "massive_plan_assumption": "Options Starter",
    "nominal_options_history_years": OPTIONS_HISTORY_YEARS,
    "nominal_history_start_on_run_date": str(nominal_history_start.date()),
    "primary_rq1_coverage": PRIMARY_RQ1_COVERAGE,
    "primary_rq2_min_bps": PRIMARY_RQ2_MIN_BPS,
    "primary_rq3_required": True,
    "primary_contract_specification": "0DTE nearest available ATM strike",
    "sensitivity_strike_offsets_points": STRIKE_OFFSETS_POINTS,
    "option_bar_window_et": [
        OPTION_BAR_START_ET,
        OPTION_BAR_END_ET,
    ],
    "candidate_sessions_total": int(len(candidates)),
    "candidate_sessions_within_nominal_history": int(len(eligible_candidates)),
    "selected_contract_rows": int(len(selected_rows)),
    "unique_contracts_requested": int(len(unique_contracts)),
    "option_bar_rows_downloaded": int(len(option_bars)),
    "historical_quotes_used": False,
    "historical_trades_used": False,
    "execution_note": (
        "Minute aggregates are trade-derived OHLC/VWAP. "
        "Historical bid/ask quotes are not available on Options Starter. "
        "RQ5 backtest must use explicit transaction-cost/slippage sensitivity."
    ),
}

manifest_path = RQ5_ROOT / "rq5_retrieval_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, default=str),
    encoding="utf-8",
)

print("Saved outputs:")
for path in [
    bars_path,
    bars_csv_path,
    bar_log_path,
    quality_path,
    manifest_path,
]:
    print(" -", path)

## 12. What the next RQ5 backtest will do

After this retrieval notebook has been run successfully, the backtest notebook will use only the saved contract map and option bars.

### Primary strategy

On a primary candidate session:

1. Use the RQ1 direction to choose Call or Put.
2. Use the same-day-expiry nearest-ATM contract.
3. Enter only after the 15:00 decision point.
4. Exit before/at the end of the final-hour window using a deterministic bar-selection rule.
5. Apply the same execution assumptions to all strategies.

### Required comparator

Use the **same candidate dates and same contract-selection method**, but take direction from the 60-minute mean-reversion rule rather than RQ1. This is essential because mean reversion outperformed RQ1 directionally in the earlier analysis.

### Transaction-cost sensitivity

Because Options Starter does not contain historical quotes, the backtest will report multiple execution scenarios rather than pretend the trade-derived aggregate price was an actual executable mid/bid/ask:

- frictionless aggregate-price reference;
- modest synthetic spread/slippage;
- medium synthetic spread/slippage;
- severe synthetic spread/slippage;
- per-contract commission/fee sensitivity.

### Performance outputs

The final RQ5 notebook will report:

- number of trades;
- win rate;
- average and median dollar P&L;
- total P&L;
- return on premium;
- profit factor;
- maximum drawdown;
- Sharpe/Sortino-style trade-return summaries;
- performance before and after cost assumptions;
- ML versus mean-reversion comparator;
- ATM versus 5/10/15/20/25/30-point OTM strike sensitivity;
- premium required per contract and capital required at entry;
- return on premium and dollar P&L by strike distance;
- probability of near-total premium loss by strike distance;
- performance by volatility regime where sample sizes permit.

Do **not** change the primary 30% / 20-bps / RQ3 filter after observing option P&L. The 25-bps and 30-bps RQ2 thresholds should remain sensitivity analyses only. Likewise, farther-OTM strikes should not replace ATM as the primary specification based solely on whichever strike produces the best backtest.